In [11]:
import time
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

pd.set_option("display.max_columns", 30)

In [14]:
import re
from pathlib import Path
from sqlalchemy import create_engine, URL, text

def mssql_url(database):
    return URL.create("mssql+pymssql", username="sa", password="DataEng#2026",
                      host="localhost", port=1433, database=database)

# 1) Run the setup script (CREATE DATABASE needs autocommit)
setup_sql = Path("../sql/01_setup_database.sql").read_text()
batches = [b.strip() for b in re.split(r"^\s*GO\s*$", setup_sql, flags=re.M | re.I) if b.strip()]

admin = create_engine(mssql_url("master"), isolation_level="AUTOCOMMIT")
with admin.connect() as conn:
    for b in batches:
        conn.execute(text(b))

# 2) Engine for the rest of the project
engine = create_engine(mssql_url("GlobalSuperstore"))
with engine.connect() as conn:
    print(conn.execute(text("SELECT DB_NAME(), SCHEMA_ID('bronze'), SCHEMA_ID('silver'), SCHEMA_ID('gold')")).fetchone())

('GlobalSuperstore', 5, 6, 7)


In [ ]:
def run_sql_file(path, engine, autocommit=False):
    """Run a .sql file that uses GO as batch separator."""
    sql = Path(path).read_text(encoding="utf-8")
    batches = [b.strip() for b in sql.split("\nGO") if b.strip()]
    conn = engine.connect()
    if autocommit:
        conn = conn.execution_options(isolation_level="AUTOCOMMIT")
    with conn:
        for batch in batches:
            if batch.upper().startswith("USE "):      # we connect to the right DB instead
                continue
            conn.execute(text(batch))
        if not autocommit:
            conn.commit()


engine = get_engine(DB_NAME)
run_sql_file("../sql/01_setup_database.sql", engine, autocommit=True)

with engine.connect() as conn:
    print(pd.read_sql("SELECT name FROM sys.schemas WHERE name IN ('bronze','silver','gold')", conn))

     name
0  bronze
1    gold
2  silver


In [ ]:
orders = pd.read_csv("/workspaces/Global_Superstore-/data/super_store/Global_Superstore2.csv", encoding="latin-1")
print(orders.shape)
orders.head()

(51290, 24)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,City,State,Country,Postal Code,Market,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Cost,Order Priority
0,32298,CA-2012-124891,31-07-2012,31-07-2012,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,United States,10024.0,US,East,TEC-AC-10003033,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...,2309.650,7,0.0,762.1845,933.57,Critical
1,26341,IN-2013-77878,05-02-2013,07-02-2013,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,Australia,NaN,APAC,Oceania,FUR-CH-10003950,Furniture,Chairs,"Novimex Executive Leather Armchair, Black",3709.395,9,0.1,-288.7650,923.63,Critical
2,25330,IN-2013-71249,17-10-2013,18-10-2013,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,Australia,NaN,APAC,Oceania,TEC-PH-10004664,Technology,Phones,"Nokia Smart Phone, with Caller ID",5175.171,9,0.1,919.9710,915.49,Medium
3,13524,ES-2013-1579342,28-01-2013,30-01-2013,First Class,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,Germany,NaN,EU,Central,TEC-PH-10004583,Technology,Phones,"Motorola Smart Phone, Cordless",2892.510,5,0.1,-96.5400,910.16,Medium
4,47221,SG-2013-4320,05-11-2013,06-11-2013,Same Day,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,Senegal,NaN,Africa,Africa,TEC-SHA-10000501,Technology,Copiers,"Sharp Wireless Fax, High-Speed",2832.960,8,0.0,311.5200,903.04,Critical


In [ ]:
API_FOLDER = "/workspaces/Global_Superstore-/data/api/"


def get_json(url, file_name, params=None):
    path = API_FOLDER + file_name + ".json"
    if os.path.exists(path):                       # already downloaded -> just read the file
        with open(path) as f:
            return json.load(f)
    for attempt in range(1, 6):                    # free APIs are sometimes slow -> try up to 5 times
        try:
            response = requests.get(url, params=params, timeout=180)
            response.raise_for_status()
            data = response.json()
            break
        except requests.exceptions.RequestException as error:
            print(f"  {file_name}: attempt {attempt} failed ({error.__class__.__name__})")
            if attempt == 5:
                raise
            time.sleep(15 * attempt)               # wait 15s, 30s, 45s ... before trying again
    with open(path, "w") as f:
        json.dump(data, f)
    print("  downloaded:", file_name)
    time.sleep(1)                                  # be polite to free APIs
    return data

In [ ]:
os.makedirs(API_FOLDER, exist_ok=True)

# 1. delete the saved error answer from the old API
old_file = API_FOLDER + "restcountries_all.json"
if os.path.exists(old_file):
    os.remove(old_file)

# 2. country data from GitHub (same fields as REST Countries, free, no key)
data = get_json("https://raw.githubusercontent.com/mledoze/countries/master/countries.json", "countries_github")

# 3. population from the World Bank (the GitHub file has no population)
pop = get_json("https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL", "worldbank_population",
               {"date": "2023", "format": "json", "per_page": 20000})
population = {row["countryiso3code"]: row["value"] for row in pop[1] if row["value"]}

countries = pd.DataFrame([{
    "name_common": c["name"]["common"],
    "name_official": c["name"]["official"],
    "cca3": c["cca3"],
    "alt_spellings": "|".join(c.get("altSpellings", [])),
    "region": c.get("region"),
    "subregion": c.get("subregion"),
    "population": population.get(c["cca3"]),
    "lat": c["latlng"][0] if c.get("latlng") else None,
    "lng": c["latlng"][1] if c.get("latlng") else None,
    "currency_code": list(c["currencies"])[0] if c.get("currencies") else None,
} for c in data])
print(countries.shape)
countries.head()

(250, 10)


,name_common,name_official,cca3,alt_spellings,region,subregion,population,lat,lng,currency_code
0,Aruba,Aruba,ABW,AW,Americas,Caribbean,107359.0,12.500000,-69.966667,AWG
1,Afghanistan,Islamic Republic of Afghanistan,AFG,AF|Afġānistān,Asia,Southern Asia,41454761.0,33.000000,65.000000,AFN
2,Angola,Republic of Angola,AGO,AO|República de Angola|ʁɛpublika de an'ɡɔla,Africa,Middle Africa,36749906.0,-12.500000,18.500000,AOA
3,Anguilla,Anguilla,AIA,AI,Americas,Caribbean,NaN,18.250000,-63.166667,XCD
4,Åland Islands,Åland Islands,ALA,AX|Aaland|Aland|Ahvenanmaa,Europe,Northern Europe,NaN,60.116667,19.900000,EUR


In [ ]:
countries.head()

,name_common,name_official,cca3,alt_spellings,region,subregion,population,lat,lng,currency_code
0,Aruba,Aruba,ABW,AW,Americas,Caribbean,107359.0,12.500000,-69.966667,AWG
1,Afghanistan,Islamic Republic of Afghanistan,AFG,AF|Afġānistān,Asia,Southern Asia,41454761.0,33.000000,65.000000,AFN
2,Angola,Republic of Angola,AGO,AO|República de Angola|ʁɛpublika de an'ɡɔla,Africa,Middle Africa,36749906.0,-12.500000,18.500000,AOA
3,Anguilla,Anguilla,AIA,AI,Americas,Caribbean,NaN,18.250000,-63.166667,XCD
4,Åland Islands,Åland Islands,ALA,AX|Aaland|Aland|Ahvenanmaa,Europe,Northern Europe,NaN,60.116667,19.900000,EUR


In [ ]:
--- Made a join 

orders.merge(df_main. on= 'columns_name')

SyntaxError: invalid syntax (4171564740.py, line 1)

In [ ]:
df=orders.head()

In [ ]:
df.to_sql("superstore_orders", engine, if_exists="replace", index=False, schema="bronze")

5

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,City,State,Country,Postal Code,Market,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Cost,Order Priority
0,32298,CA-2012-124891,31-07-2012,31-07-2012,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,United States,10024.0,US,East,TEC-AC-10003033,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...,2309.650,7,0.0,762.1845,933.57,Critical
1,26341,IN-2013-77878,05-02-2013,07-02-2013,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,Australia,NaN,APAC,Oceania,FUR-CH-10003950,Furniture,Chairs,"Novimex Executive Leather Armchair, Black",3709.395,9,0.1,-288.7650,923.63,Critical
2,25330,IN-2013-71249,17-10-2013,18-10-2013,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,Australia,NaN,APAC,Oceania,TEC-PH-10004664,Technology,Phones,"Nokia Smart Phone, with Caller ID",5175.171,9,0.1,919.9710,915.49,Medium
3,13524,ES-2013-1579342,28-01-2013,30-01-2013,First Class,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,Germany,NaN,EU,Central,TEC-PH-10004583,Technology,Phones,"Motorola Smart Phone, Cordless",2892.510,5,0.1,-96.5400,910.16,Medium
4,47221,SG-2013-4320,05-11-2013,06-11-2013,Same Day,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,Senegal,NaN,Africa,Africa,TEC-SHA-10000501,Technology,Copiers,"Sharp Wireless Fax, High-Speed",2832.960,8,0.0,311.5200,903.04,Critical
